In [4]:
import pandas as pd
import numpy as np

In [5]:
#read csv
df= pd.read_csv('train_cleaned.csv')

In [6]:
# Validate risk scoring framework
risk_validation = df.groupby('lapse_risk_score')['renewal_status'].agg(
    total_customer='count',
    renewal_rate= lambda x: round((x.mean())*100,2),
    lapse_rate = lambda x:round((1-x.mean())*100,2)).reset_index()
print(risk_validation)
        

  lapse_risk_score  total_customer  renewal_rate  lapse_rate
0             High           46991         31.95       68.05
1              Low           98585         89.17       10.83
2           Medium          235533         64.10       35.90


In [14]:
#Revenue at Risk per tier 

Revenue_risk_per_tier= df[df['renewal_status'] == 0].groupby('lapse_risk_score')['Annual_Premium'].sum().round(2).reset_index()
Revenue_risk_per_tier['Annual_Premium'] = Revenue_risk_per_tier['Annual_Premium'].apply(lambda x: f"{x:,.2f}") # so that we can show the full value in sum and not 123 e+008
print(Revenue_risk_per_tier)

  lapse_risk_score    Annual_Premium
0             High    942,228,741.00
1              Low    318,218,564.50
2           Medium  2,546,870,238.00


In [16]:
priority_table = pd.DataFrame({
    'Risk_Tier': ['High', 'Medium', 'Low'],
    'Customer_Count': [46991, 235533, 98585],
    'Lapse_Rate': ['68.05%', '35.90%', '10.83%'],
    'Revenue_at_Risk': ['$942,228,741', '$2,546,870,238', '$318,218,564'],
    'Recommended_Action': [
        'Urgent agent call -- high lapse rate, focus on win-back offers',
        'Immediate agent call + renewal discount for high premium segment',
        'Automated renewal reminder email'
    ],
    'Campaign_Priority': ['High', 'Critical', 'Low']
})

print(priority_table.to_string(index=False))

Risk_Tier  Customer_Count Lapse_Rate Revenue_at_Risk                                               Recommended_Action Campaign_Priority
     High           46991     68.05%    $942,228,741   Urgent agent call -- high lapse rate, focus on win-back offers              High
   Medium          235533     35.90%  $2,546,870,238 Immediate agent call + renewal discount for high premium segment          Critical
      Low           98585     10.83%    $318,218,564                                 Automated renewal reminder email               Low


# Risk Scoring Framework
The risk scoring framework was validated in Python, confirming clear separation between tiers. Medium risk customers represent the highest priority despite a lower individual lapse rate of 35.90%, because their sheer volume of 235,533 customers translates to 2.54 billion in revenue at risk -- nearly 3x the High risk tier. High risk customers lapse at 68.05% making recovery difficult, but still warrant urgent agent outreach for win-back efforts. Low risk customers require only automated email reminders to maintain their already strong 89.17% renewal rate. Budget and effort should be concentrated on the Medium risk segment where intervention is most likely to succeed and revenue recovery is highest.